# Transfer Learning for Manufacturing Defect Detection with ResNet18

**Goal**  
Teach transfer learning end‑to‑end by fine‑tuning a pre‑trained ResNet18 for binary image classification (normal vs. defective) on a **very small, class‑imbalanced** industrial dataset.

**Input**  
Directory of 224×224 RGB images organised as `train/normal/`, `train/defective/`, `val/normal/`, `val/defective/`.  
If the directory is empty, the notebook **automatically downloads the Hymenoptera dataset** and redistributes it to simulate 400 training / 100 validation images with an 80% / 20% class imbalance – using only real photographs, **no dummy/synthetic data**.

**Output**  
- 4 trained model artefacts (`.pth`):
  - Full‑freeze (only new FC trained)
  - Layer4‑unfreeze (conv blocks 4 unfrozen)
  - PEFT / LoRA‑adapted
  - From‑scratch baseline (randomly initialised)
- Training logs (per‑epoch loss & accuracy) for each variant.
- Comparison table & plot (accuracy, precision/recall per class, overfitting gap).

**Constraints & Trade‑offs**  
- This notebook **runs on CPU**; training is reduced to **5 epochs** per variant so the 4‑way comparison finishes in a reasonable time.
- **If you have a GPU**, increase `NUM_EPOCHS` for more statistically meaningful results.
- The dataset is heavily imbalanced (80/20); we handle this with a **weighted loss** and discuss its effect.

## Section 0 – Setup & Framing

In [ ]:
# ------------------------------ Section 0 : Imports & Environment ------------------------------
import os, copy, random, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms, models
from torchvision.models.resnet import ResNet18_Weights
from torchvision.datasets.utils import download_url, extract_archive

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# PEFT library (optional) – fallback to manual LoRA if not available / crashes
try:
    from peft import LoraConfig, get_peft_model
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False
    print("⚠️  peft not installed. Will use manual LoRA implementation.")

# Manual LoRA fallback (always defined, used when peft fails)
class LoraConv2d(nn.Module):
    """
    Low‑rank adaptation wrapper for Conv2d.
    Adds a trainable low‑rank path: output = original_conv(x) + lora_up(lora_down(x)).
    Only used for stride‑1 convolutions to avoid spatial size mismatch.
    """
    def __init__(self, conv: nn.Conv2d, rank=4):
        super().__init__()
        # freeze original conv
        self.conv = conv
        for param in self.conv.parameters():
            param.requires_grad = False
        self.rank = rank
        # Low‑rank bottleneck: 1×1 conv reduce, then expand
        self.lora_down = nn.Conv2d(conv.in_channels, rank, kernel_size=1, bias=False)
        self.lora_up   = nn.Conv2d(rank, conv.out_channels, kernel_size=1, bias=False)
        nn.init.kaiming_uniform_(self.lora_down.weight, a=np.sqrt(5))
        nn.init.zeros_(self.lora_up.weight)

    def forward(self, x):
        # original conv output (frozen)
        with torch.no_grad():
            orig_out = self.conv(x)
        # LoRA path
        lora_out = self.lora_up(self.lora_down(x))
        return orig_out + lora_out

def inject_manual_lora(module, rank=4, skip_first_conv=True):
    """
    Recursively replace stride‑1 Conv2d layers with LoraConv2d.
    Skip conv1 (first layer) and any conv with stride > 1.
    """
    for name, child in module.named_children():
        if isinstance(child, nn.Conv2d):
            # skip first conv (7x7 stride 2) and any non‑stride‑1 conv
            if (skip_first_conv and name == "conv1") or child.stride[0] != 1:
                continue
            setattr(module, name, LoraConv2d(child, rank=rank))
        else:
            inject_manual_lora(child, rank=rank, skip_first_conv=False)

# Device detection – favour GPU, fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Section 1 – The Pre‑trained Model

We load a ResNet18 pre‑trained on ImageNet using the **current `weights` API** (no deprecated `pretrained=True`).  
Let’s inspect its architecture and parameters.

In [ ]:
# ------------------------------ Section 1 : Load pre-trained ResNet18 ------------------------------
model_pretrained = models.resnet18(weights=ResNet18_Weights.DEFAULT).to(DEVICE)
print(model_pretrained)

In [ ]:
# Inspect top‑level children (blocks)
for name, child in model_pretrained.named_children():
    print(f"{name:15} -> {type(child).__name__}")

In [ ]:
# Show a few parameters (conv1.weight, etc.)
for name, param in list(model_pretrained.named_parameters())[:6]:
    print(f"{name:40} shape {tuple(param.shape)}")

**What does “pre‑trained” mean?**  
- The weights are the result of training ResNet18 on **ImageNet** – 1.28 million natural images from 1,000 object categories.  
- Early layers learn generic features (edges, textures, colours); later layers capture higher‑level patterns specific to ImageNet classes.  
- **Why do these features generalise?** Low‑level statistics (edges, corners) are common across most natural images, and even higher‑level concepts (shapes, parts) can be reused for many vision tasks.  
- However, the final fully‑connected layer is strictly tied to the 1,000 ImageNet classes – we will replace it.

## Section 2 – Source Task vs. Target Task

We now set up our tiny **manufacturing defect** dataset.  
If no custom data is provided, the notebook will automatically download the Hymenoptera dataset (real photographs of ants & bees) and redistribute the images to create the required 80%/20% class imbalance.

In [ ]:
# ------------------------------ Section 2 : Dataset preparation ------------------------------
data_root = Path("./data")
train_dir = data_root / "train"
val_dir   = data_root / "val"

def folder_has_images(folder):
    """Return True if the folder contains at least one image file."""
    if not folder.exists():
        return False
    extensions = {".jpg", ".jpeg", ".png", ".ppm", ".bmp"}
    for f in folder.rglob("*"):
        if f.suffix.lower() in extensions:
            return True
    return False

if (folder_has_images(train_dir / "normal") and folder_has_images(train_dir / "defective") and
    folder_has_images(val_dir / "normal") and folder_has_images(val_dir / "defective")):
    print("✅ Custom dataset found. Using existing folder structure.")
else:
    print("⚠️  No dataset found. Downloading Hymenoptera dataset (real photos) …")
    hymenoptera_root = data_root / "hymenoptera_data"
    zip_path = data_root / "hymenoptera_data.zip"

    try:
        url = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
        # download_url(root, filename) – no 'progress' argument in recent torchvision
        download_url(url, str(data_root), "hymenoptera_data.zip")
        extract_archive(str(zip_path), str(data_root))
        os.remove(zip_path)
        print("✅ Download successful.")
    except Exception as e:
        raise RuntimeError(
            f"Could not download Hymenoptera dataset automatically.\n"
            f"Please manually download from {url} and extract it to {hymenoptera_root}\n"
            f"Error: {e}"
        )

    # Collect all real images
    all_images = []
    for split in ["train", "val"]:
        split_dir = hymenoptera_root / split
        if not split_dir.exists():
            continue
        for cls in os.listdir(split_dir):
            class_dir = split_dir / cls
            if class_dir.is_dir():
                all_images.extend(class_dir.iterdir())
    all_images = [p for p in all_images if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    print(f"Found {len(all_images)} real images in Hymenoptera dataset.")

    # Target sizes: 400 train, 100 val with 80% 'normal', 20% 'defective'
    N_TRAIN, N_VAL = 400, 100
    random.shuffle(all_images)
    sampled_train = random.choices(all_images, k=N_TRAIN)
    sampled_val   = random.choices(all_images, k=N_VAL)

    n_train_normal = int(0.8 * N_TRAIN)
    n_val_normal   = int(0.8 * N_VAL)

    train_labels = ["normal"] * n_train_normal + ["defective"] * (N_TRAIN - n_train_normal)
    val_labels   = ["normal"] * n_val_normal   + ["defective"] * (N_VAL   - n_val_normal)
    random.shuffle(train_labels)
    random.shuffle(val_labels)

    # Clear old structure and create folders
    for split in ["train", "val"]:
        for cls in ["normal", "defective"]:
            shutil.rmtree(data_root / split / cls, ignore_errors=True)
            (data_root / split / cls).mkdir(parents=True)

    # Copy images to target folders, avoiding name collisions
    for split, samples, labels in zip(["train", "val"],
                                     [sampled_train, sampled_val],
                                     [train_labels, val_labels]):
        for src_path, lbl in zip(samples, labels):
            src = Path(src_path)
            dst = data_root / split / lbl / src.name
            counter = 1
            while dst.exists():
                stem = src.stem + f"_{counter}"
                dst = data_root / split / lbl / (stem + src.suffix)
                counter += 1
            shutil.copy2(src, dst)

    print("✅ Real‑image dataset created with exact 80/20 imbalance (400 train, 100 val).")


In [ ]:
# Load the dataset using ImageFolder
train_dataset = datasets.ImageFolder(root=str(train_dir))
val_dataset   = datasets.ImageFolder(root=str(val_dir))
print(f"Train samples: {len(train_dataset)}, classes: {train_dataset.classes}")
print(f"Val   samples: {len(val_dataset)}")
print(f"Class distribution (train): {np.bincount(train_dataset.targets)}")
print(f"Class distribution (val):   {np.bincount(val_dataset.targets)}")

### Source ↔ Target comparison

| Aspect | Source (ImageNet) | Target (Defect Detection) |
|--------|------------------|---------------------------|
| **Number of classes** | 1,000 | 2 |
| **Domain** | Natural photographs | Industrial (manufactured parts, close‑ups) |
| **Typical objects** | Animals, vehicles, everyday items | Metal surfaces, scratches, dents |
| **Image statistics** | Varied lighting, backgrounds | Controlled lighting, repetitive textures |

**Visual similarity judgement** – Stop here and look at a few target images.  
ImageNet features *should* transfer reasonably well because low‑level edges and textures still exist. However, the target domain may contain fine‑grained defects that differ from natural categories, implying **some domain shift**. This will affect how aggressively we fine‑tune later layers.

## Section 3 – Feature Reuse: Visualising What Transfers

In [ ]:
# ------------------------------ Section 3 : Activation visualisation (corrected) ------------------------------
# Hook into an early layer (layer1[0].conv1) and a late layer (layer4[1].conv2)
activations = {}
def get_activation(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

model_pretrained.layer1[0].conv1.register_forward_hook(get_activation('early'))
model_pretrained.layer4[1].conv2.register_forward_hook(get_activation('late'))

# Manually load one image (avoid DataLoader collate error with PIL images)
sample_path = list((train_dir / "normal").glob("*"))[0]  # any real image
img_pil = Image.open(sample_path).convert("RGB")

# Apply the same transforms the pre‑trained model expects
normalise = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
img_tensor = transforms.ToTensor()(img_pil)          # [0,1]
img_tensor = normalise(img_tensor).unsqueeze(0)      # (1, C, H, W)

model_pretrained.eval()
with torch.no_grad():
    _ = model_pretrained(img_tensor.to(DEVICE))

# Inverse‑normalise for display
inv_normalise = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225])
img_display = inv_normalise(img_tensor.squeeze(0)).cpu().clamp(0, 1)

plt.figure(figsize=(10, 4))
plt.subplot(1, 3, 1)
plt.imshow(img_display.permute(1, 2, 0))
plt.title("Input image"); plt.axis("off")

# Show the first activation channel for each layer (representative)
plt.subplot(1, 3, 2)
plt.imshow(activations['early'][0, 0].cpu(), cmap='viridis')
plt.title("Early layer (ch.0)"); plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(activations['late'][0, 0].cpu(), cmap='viridis')
plt.title("Late layer (ch.0)"); plt.axis("off")

plt.suptitle("Early vs. late activation maps", fontsize=14)
plt.tight_layout()
plt.show()

**Observations**  
- **Early layers** (conv1, layer1) react to edges, colour blobs, and simple textures – these are largely domain‑agnostic.  
- **Late layers** (layer4) respond to more complex, object‑specific patterns. When we fine‑tune, we must decide how far to let ImageNet knowledge be overwritten.

## Section 4 – Network Modification

We replace the final fully‑connected layer to output 2 classes instead of 1,000.  
This is the **canonical first step** in any transfer learning pipeline.

In [ ]:
# ------------------------------ Section 4 : FC layer replacement ------------------------------
NUM_CLASSES = 2
def replace_fc(model):
    """Replace the final FC layer of a ResNet model with a new one for binary classification."""
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, NUM_CLASSES)
    return model

## Section 5 – Freeze vs. Fine‑tune vs. PEFT (Core Comparison)

We define **three isolated training functions**. Each function:
- Loads a **fresh** ResNet18 with ImageNet weights.
- Replaces the FC layer.
- Applies a different freezing/adaptation strategy.
- Trains using the same data, augmentations, and weighted loss (Section 6).

This guarantees a **fair, isolated comparison**.

#### Strategy 1: Full Freeze
Freeze **all** convolutional layers → only the new FC head is trained.  
Best when the target dataset is extremely small and the feature extractor is already very good.

In [ ]:
def train_full_freeze(train_loader, val_loader, num_epochs=5):
    """Freeze all convolutional layers; train only the FC head."""
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    model = replace_fc(model).to(DEVICE)
    # Freeze everything except fc
    for name, param in model.named_parameters():
        if "fc" not in name:
            param.requires_grad = False
    return train_model(model, train_loader, val_loader, num_epochs, model_name="full_freeze")

#### Strategy 2: Unfreeze Layer4
Freeze early layers but **unfreeze the last residual block (layer4)** plus the new FC.  
This allows the highest‑level features to adapt to defects while keeping the backbone stable.  
It is a pragmatic compromise when the dataset is very small but some domain adaptation is needed.

In [ ]:
def train_unfreeze_layer4(train_loader, val_loader, num_epochs=5):
    """Freeze all conv layers except layer4; train layer4 + FC."""
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    model = replace_fc(model).to(DEVICE)
    # Freeze all, then unfreeze layer4 & fc
    for name, param in model.named_parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if "layer4" in name or "fc" in name:
            param.requires_grad = True
    return train_model(model, train_loader, val_loader, num_epochs, model_name="layer4_unfreeze")

#### Strategy 3: PEFT (LoRA)
Inject **low‑rank adapters** into the convolutional layers while keeping original weights frozen.  
If the `peft` library is unavailable or fails at runtime, a manual LoRA implementation is used.  
**Important:** The manual fallback only wraps stride‑1 Conv2d layers (skipping `conv1` and downsampling layers) to avoid spatial size mismatches.

In [ ]:
def train_peft(train_loader, val_loader, num_epochs=5):
    """Apply LoRA (via peft or manual fallback) to a frozen ResNet18 backbone."""
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    model = replace_fc(model).to(DEVICE)
    
    peft_used = False
    if PEFT_AVAILABLE:
        try:
            config = LoraConfig(
                r=4,
                lora_alpha=8,
                target_modules=["conv1", "conv2"],  # typical for ResNet conv layers
                lora_dropout=0.1,
                bias="none",
                modules_to_save=["fc"],  # train fc normally
            )
            model = get_peft_model(model, config)
            peft_used = True
            print("✅ Using peft library for LoRA.")
        except Exception as e:
            print(f"⚠️  peft runtime error: {e}")
            print("   Falling back to manual LoRA implementation.")

    if not peft_used:
        # Manual LoRA: freeze all parameters first, then inject trainable LoRA adapters
        for param in model.parameters():
            param.requires_grad = False
        # Inject only into stride‑1 convs (skipping conv1, downsampling layers)
        inject_manual_lora(model, rank=4)
        # Unfreeze FC head
        for param in model.fc.parameters():
            param.requires_grad = True
        print("✅ Using manual LoRA implementation (stride‑1 convs only).")

    return train_model(model, train_loader, val_loader, num_epochs, model_name="peft_lora")

## Section 6 – Training Loop with Class Imbalance Handling

We build a **single reusable training loop** that all Section 5 functions call.  
Key design choices:
- **Weighted CrossEntropyLoss**: inverse‑frequency weights to counter the 80/20 imbalance.
- **Aggressive data augmentation** for the training set; minimal/deterministic for validation.
- **Per‑epoch logging** of loss and accuracy on both train and val.

In [ ]:
# ------------------------------ Section 6 : Data transforms & augmentations ------------------------------
IMG_SIZE = 224
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Apply transforms to datasets (before DataLoader)
train_dataset.transform = train_transforms
val_dataset.transform   = val_transforms

# DataLoaders (small batch for CPU)
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Compute class weights for weighted loss
class_counts = np.bincount(train_dataset.targets)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)  # normalise
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"Class weights: {class_weights}")

In [ ]:
# ------------------------------ Section 6 : Reusable training loop ------------------------------
def train_model(model, train_loader, val_loader, num_epochs, model_name="model"):
    """
    Generic training loop with weighted cross‑entropy loss.
    Returns (trained_model, log_dict) where log_dict contains per‑epoch loss/accuracy.
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    # Only optimise parameters that require grad
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    log = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        epoch_train_loss = running_loss / total
        epoch_train_acc = correct / total
        log["train_loss"].append(epoch_train_loss)
        log["train_acc"].append(epoch_train_acc)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        log["val_loss"].append(epoch_val_loss)
        log["val_acc"].append(epoch_val_acc)

        print(f"{model_name} | Epoch {epoch:2d}/{num_epochs}: "
              f"Train Loss {epoch_train_loss:.4f} Acc {epoch_train_acc:.4f} | "
              f"Val Loss {epoch_val_loss:.4f} Acc {epoch_val_acc:.4f}")

        # Save best model
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), f"{model_name}.pth")
    print(f"✅ Training completed. Best val accuracy: {best_val_acc:.4f}\n")
    # Load best weights for return
    model.load_state_dict(torch.load(f"{model_name}.pth", map_location=DEVICE))
    return model, log

### Orchestrating Cell (End of Section 6)
Now we run all three transfer‑learning variants **sequentially** using the same data and training settings.

In [ ]:
NUM_EPOCHS = 5  # reduced for CPU – increase if you have a GPU

print("=" * 50)
print("  1. Training FULL FREEZE variant")
print("=" * 50)
model_full_freeze, log_full_freeze = train_full_freeze(train_loader, val_loader, num_epochs=NUM_EPOCHS)

print("=" * 50)
print("  2. Training LAYER4 UNFREEZE variant")
print("=" * 50)
model_layer4, log_layer4 = train_unfreeze_layer4(train_loader, val_loader, num_epochs=NUM_EPOCHS)

print("=" * 50)
print("  3. Training PEFT / LoRA variant")
print("=" * 50)
model_peft, log_peft = train_peft(train_loader, val_loader, num_epochs=NUM_EPOCHS)

## Section 7 – Baseline, Evaluation & QA

We train a **from‑scratch** ResNet18 (randomly initialised) as a control.  
Then we compute classification metrics for all four variants and compare them.

In [ ]:
# ------------------------------ Section 7 : From‑scratch baseline ------------------------------
def train_from_scratch(train_loader, val_loader, num_epochs=5):
    """Train a randomly initialised ResNet18 with no pre‑training."""
    model = models.resnet18(weights=None)  # random init
    model = replace_fc(model).to(DEVICE)
    return train_model(model, train_loader, val_loader, num_epochs, model_name="from_scratch")

print("=" * 50)
print("  4. Training FROM SCRATCH baseline")
print("=" * 50)
model_scratch, log_scratch = train_from_scratch(train_loader, val_loader, num_epochs=NUM_EPOCHS)

In [ ]:
# ------------------------------ Section 7 : Evaluation helper ------------------------------
def evaluate_model(model, loader, device=DEVICE):
    """Return true labels, predictions for the full validation set."""
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

def compute_metrics(labels, preds, class_names):
    """Return dict of accuracy, per‑class precision/recall, and overfitting gap placeholder."""
    report = classification_report(labels, preds, target_names=class_names,
                                    output_dict=True, zero_division=0)
    acc = report["accuracy"]
    prec_normal = report["normal"]["precision"]
    rec_normal  = report["normal"]["recall"]
    prec_defect = report["defective"]["precision"]
    rec_defect  = report["defective"]["recall"]
    return {
        "accuracy": acc,
        "precision_normal": prec_normal,
        "recall_normal": rec_normal,
        "precision_defective": prec_defect,
        "recall_defective": rec_defect
    }

# Evaluate all four variants
variants = {
    "Full Freeze":   (model_full_freeze, log_full_freeze),
    "Layer4 Unfreeze": (model_layer4, log_layer4),
    "PEFT / LoRA":   (model_peft, log_peft),
    "From Scratch":  (model_scratch, log_scratch)
}

results = {}
for name, (m, log) in variants.items():
    labels, preds = evaluate_model(m, val_loader)
    met = compute_metrics(labels, preds, ["normal", "defective"])
    # Overfitting gap: difference between last epoch train and val accuracy
    train_acc_last = log["train_acc"][-1]
    val_acc_last   = log["val_acc"][-1]
    met["overfitting_gap"] = train_acc_last - val_acc_last
    results[name] = met
    print(f"\n{name}:")
    print(f"  Accuracy         : {met['accuracy']:.4f}")
    print(f"  Precision (normal): {met['precision_normal']:.4f}  Recall (normal): {met['recall_normal']:.4f}")
    print(f"  Precision (defect): {met['precision_defective']:.4f}  Recall (defect): {met['recall_defective']:.4f}")
    print(f"  Overfitting gap   : {met['overfitting_gap']:.4f}")

In [ ]:
# ------------------------------ Section 7 : Comparison table & plot ------------------------------
df = pd.DataFrame(results).T
df = df[["accuracy", "precision_normal", "recall_normal",
         "precision_defective", "recall_defective", "overfitting_gap"]]
print("\n📊 Comparison Table")
print(df.round(4))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df[["accuracy", "overfitting_gap"]].plot(kind="bar", ax=ax[0], colormap="Set2")
ax[0].set_title("Validation Accuracy & Overfitting Gap")
ax[0].set_ylabel("Score")

df[["precision_normal", "recall_normal", "precision_defective", "recall_defective"]].plot(
    kind="bar", ax=ax[1], colormap="Paired")
ax[1].set_title("Per‑class Precision & Recall")
ax[1].set_ylabel("Score")
plt.tight_layout()
plt.show()

**Sanity check – negative transfer?**  
If any transfer variant performs *worse* than the from‑scratch baseline, we have a case of **negative transfer**.  
That could happen when the source and target domains are too dissimilar (see Section 8).  
Examine the table above: did the pre‑trained features help or hurt?

## Section 8 – Reflection: When Would Transfer Fail?

We assumed that ImageNet features would transfer usefully because industrial images still contain edges and textures.  
But consider a target task that is **radically different**:

- **Radio‑frequency (RF) spectrograms** – no natural edges, colour, or object shapes.
- **Medical volumetric scans** (CT/MRI) – grey‑scale, 3D structures, completely different statistics.
- **Astronomical images** – point sources, non‑photographic noise.

In such cases, the low‑level filters learned on ImageNet (Gabor‑like edges, colour blobs) might be **irrelevant or even misleading**, leading to negative transfer.  
The risk is highest when the target data distribution lies far from the natural‑image manifold.  
This is why, in practice, we always **validate the assumption** of domain similarity before freezing layers – exactly what Section 2’s visual similarity check was meant to surface.

**Final thought**: Transfer learning is not a silver bullet; its success depends critically on how close the source and target tasks really are.